# Treasure Maze — versione corretta

Progetto di Intelligenza Artificiale — Miglietta, Kandli, Rossi.

Questa versione sistema il notebook originale, che presentava diversi problemi che lo rendevano non funzionante:
- **quattro definizioni diverse** della classe `TreasureMaze` (due erano scheletri con metodi placeholder mai implementati, rimaste per errore nel file), che rendevano il codice illeggibile e fragile: bastava eseguire le celle in un ordine leggermente diverso per rompere tutto.
- funzioni ausiliarie (`best_first_search_graph`, `ucs`, `astar`, `misplaced_tiles`, `manh`) mai usate, che referenziavano `PriorityQueue`/`memoize` senza importarli: sarebbero andate in errore se richiamate.
- comandi `!pip`, `!apt`, `files.upload()`, `drive.mount()` mescolati alla logica del programma, che rendevano il notebook eseguibile solo su Google Colab, e solo eseguendo le celle nell'ordine esatto in cui erano state scritte.
- il conteggio delle **"celle esplorate"** nel capitolo 3 del pdf era in realtà un bug: veniva usata la lunghezza del *percorso soluzione*, non il numero di stati realmente generati durante la ricerca. Per questo A\* e UCS risultavano sempre con lo *stesso* numero di celle esplorate — un risultato che, a ben guardare, non aveva senso (UCS, non avendo euristica, esplora tipicamente più stati di A\*).
- i muri (`X`) venivano trattati come celle invalicabili, mentre la consegna li descrive come *abbattibili a costo 5* (quindi attraversabili, a un prezzo).
- la ricerca veniva lanciata come tante ricerche separate, una per ogni tesoro in ordine di lista, invece di un vero problema di ricerca che include i tesori raccolti nello stato: questo impediva del tutto la modalità **"raccogli almeno k tesori"** richiesta dalla consegna.
- `extract_maze` apriva una finestra `plt.show()` per **ogni singola cella** dell'immagine (64 finestre per un labirinto 8x8), e sovrascriveva la predizione del modello con soglie di probabilità arbitrarie (`prob_S > 0.05`), invece di fidarsi semplicemente della classe più probabile.

Il codice qui sotto è organizzato in sezioni indipendenti e ha lo stesso comportamento del pacchetto Python `treasure_maze/` (stesso codice, spiegato passo passo).

## 0. Setup
Da Colab: carica `aima.zip` quando richiesto. In locale: estrai `aima.zip` in questa cartella una volta sola, poi salta questa cella.

In [ ]:
import sys, os, zipfile

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import files
    if not os.path.exists("aima"):
        print("Carica aima.zip (distribuito su Unistudium)")
        uploaded = files.upload()
        with zipfile.ZipFile("aima.zip") as z:
            z.extractall(".")
else:
    if not os.path.exists("aima") and os.path.exists("aima.zip"):
        with zipfile.ZipFile("aima.zip") as z:
            z.extractall(".")

sys.path.append(os.path.abspath("."))
print("aima disponibile:", os.path.exists("aima"))

## 1. Il problema di ricerca `TreasureMaze`

Una sola classe, coerente con la consegna: lo stato include i tesori già raccolti e i muri già abbattuti, così una singola ricerca A*/UCS può risolvere sia "raccogli tutti i tesori" sia "raccoglierne almeno k".

In [ ]:
# -*- coding: utf-8 -*-
"""
Definizione del problema Treasure Maze come problema di ricerca AIMA.

Rispetto alla versione originale, qui:
- esiste UNA SOLA classe TreasureMaze (nel file originale ne comparivano
  quattro definizioni diverse, una sovrascritta dall'altra: due erano
  "scheletri" con metodi placeholder mai completati e mai usati, e la loro
  sola presenza rendeva il modulo enormemente più difficile da leggere e
  mantenere. Restava attiva solo l'ultima, le altre erano codice morto).
- lo stato include ESPLICITAMENTE i tesori già raccolti, così la ricerca
  può davvero risolvere l'obiettivo "raccogli tutti i tesori" o "raccogline
  almeno k", come richiesto dal progetto (capitolo 1.1 del pdf). La
  versione originale, invece, lanciava una ricerca A*/UCS separata verso
  ogni tesoro preso in ordine di lista: questo non è un vero "problema di
  ricerca con più obiettivi", non garantisce l'ottimalità del percorso
  complessivo e non permette affatto la modalità "almeno k tesori".
- i muri ('X') sono ATTRAVERSABILI abbattendoli a costo 5, come descritto
  nel dominio (capitolo 1, "X: muro, abbattibile con costo 5"). Nel codice
  originale i muri erano invece celle proibite (rimosse da actions()),
  in contraddizione con la consegna.
- il conteggio delle "celle esplorate" viene fatto correttamente
  instrumentando il problema (aima.search.InstrumentedProblem), invece di
  usare per errore la lunghezza del percorso soluzione (bug presente nel
  codice originale: veniva chiamata num_visited_cells la lunghezza di
  astar_solution.path(), che è il percorso finale, non l'insieme dei nodi
  espansi durante la ricerca).
"""
from __future__ import annotations

import time
from dataclasses import dataclass
from typing import FrozenSet, List, Optional, Tuple

from aima.search import Problem, InstrumentedProblem, astar_search, uniform_cost_search

Position = Tuple[int, int]
State = Tuple[Position, FrozenSet[Position], FrozenSet[Position]]
# state = (posizione_corrente, tesori_raccolti, muri_abbattuti)

WALL = "X"
START = "S"
TREASURE = "T"
WALL_COST = 5


class TreasureMaze(Problem):
    """Problema di ricerca: un agente si muove su una griglia NxM per
    raccogliere tesori ('T'), potendo attraversare muri ('X') abbattendoli
    a un costo fisso. Ogni cella calpestabile ha un costo di transito
    numerico (1-4). L'obiettivo è raccogliere tutti i tesori, oppure
    almeno `k` di essi se `k` è specificato.
    """

    def __init__(self, maze: List[List[str]], start: Position, k: Optional[int] = None):
        self.maze = maze
        self.rows = len(maze)
        self.cols = len(maze[0]) if self.rows else 0
        self.treasures = frozenset(
            (r, c) for r in range(self.rows) for c in range(self.cols) if maze[r][c] == TREASURE
        )
        if not self.treasures:
            raise ValueError("Nessun tesoro ('T') presente nel labirinto.")
        if k is not None and not (1 <= k <= len(self.treasures)):
            raise ValueError(f"k deve essere compreso tra 1 e {len(self.treasures)}.")
        self.k = k  # None -> raccogliere tutti i tesori

        initial: State = (start, frozenset(), frozenset())
        super().__init__(initial)

    # -- utilità -----------------------------------------------------
    def _in_bounds(self, pos: Position) -> bool:
        r, c = pos
        return 0 <= r < self.rows and 0 <= c < self.cols

    def _cell_cost(self, pos: Position, broken_walls: FrozenSet[Position]) -> int:
        r, c = pos
        value = self.maze[r][c]
        if value == WALL:
            return 1 if pos in broken_walls else WALL_COST
        if value.isdigit():
            return int(value)
        return 1  # 'S', 'T' o cella vuota: costo base

    # -- interfaccia Problem ------------------------------------------
    def actions(self, state: State) -> List[str]:
        (row, col), _, _ = state
        moves = {
            "UP": (row - 1, col),
            "DOWN": (row + 1, col),
            "LEFT": (row, col - 1),
            "RIGHT": (row, col + 1),
        }
        return [name for name, pos in moves.items() if self._in_bounds(pos)]

    def result(self, state: State, action: str) -> State:
        (row, col), collected, broken = state
        delta = {"UP": (-1, 0), "DOWN": (1, 0), "LEFT": (0, -1), "RIGHT": (0, 1)}[action]
        new_pos = (row + delta[0], col + delta[1])

        if self.maze[new_pos[0]][new_pos[1]] == WALL:
            broken = broken | {new_pos}

        if new_pos in self.treasures:
            collected = collected | {new_pos}

        return new_pos, collected, broken

    def goal_test(self, state: State) -> bool:
        _, collected, _ = state
        needed = self.k if self.k is not None else len(self.treasures)
        return len(collected) >= needed

    def path_cost(self, c, state1: State, action: str, state2: State) -> int:
        (_, collected1, broken1) = state1
        new_pos, _, _ = state2
        return c + self._cell_cost(new_pos, broken1)

    def h(self, node) -> int:
        """Euristica ammissibile: distanza di Manhattan dal tesoro
        non ancora raccolto più vicino (0 se l'obiettivo è già
        soddisfatto)."""
        (row, col), collected, _ = node.state
        remaining = self.treasures - collected
        if not remaining:
            return 0
        return min(abs(row - tr) + abs(col - tc) for tr, tc in remaining)


@dataclass
class SearchResult:
    algorithm: str
    path: Optional[List[Position]]
    cost: Optional[int]
    time_seconds: float
    states_generated: int  # celle (stati) effettivamente generati durante la ricerca
    goal_tests: int


def _run(maze: List[List[str]], start: Position, k: Optional[int], algorithm: str) -> SearchResult:
    problem = TreasureMaze(maze, start, k=k)
    instrumented = InstrumentedProblem(problem)

    t0 = time.perf_counter()
    if algorithm == "astar":
        node = astar_search(instrumented, instrumented.h)
    elif algorithm == "ucs":
        node = uniform_cost_search(instrumented)
    else:
        raise ValueError("algorithm deve essere 'astar' o 'ucs'")
    elapsed = time.perf_counter() - t0

    if node is None:
        return SearchResult(algorithm, None, None, elapsed, instrumented.states, instrumented.goal_tests)

    path = [s[0] for s in [n.state for n in node.path()]]
    return SearchResult(algorithm, path, node.path_cost, elapsed, instrumented.states, instrumented.goal_tests)


def solve_treasure_maze(
    maze: List[List[str]],
    start: Optional[Position] = None,
    k: Optional[int] = None,
) -> Tuple[SearchResult, SearchResult]:
    """Risolve il labirinto con A* e UCS e restituisce i due risultati,
    inclusi tempo di esecuzione e numero di stati generati (celle
    esplorate), per il confronto di prestazioni richiesto nel capitolo 3
    del progetto.
    """
    if start is None:
        start = find_start(maze)
    astar_result = _run(maze, start, k, "astar")
    ucs_result = _run(maze, start, k, "ucs")
    return astar_result, ucs_result


def find_start(maze: List[List[str]]) -> Position:
    for r, row in enumerate(maze):
        for c, value in enumerate(row):
            if value == START:
                return r, c
    raise ValueError("Nessuna posizione di partenza ('S') trovata nel labirinto.")


## 2. Classificazione delle celle (CNN) ed estrazione del labirinto dall'immagine

In [ ]:
# -*- coding: utf-8 -*-
"""
Classificazione delle celle del labirinto (CNN) ed estrazione della
matrice del labirinto da un'immagine.

Differenze principali rispetto all'originale:
- Nessuna dipendenza da Google Colab (niente `google.colab.files`,
  `drive.mount`, comandi `!pip`/`!apt`): il codice originale non poteva
  proprio essere eseguito al di fuori di un notebook Colab, mentre questo
  modulo funziona come normale script/libreria Python.
- I pesi del modello vengono salvati e ricaricati (`load_or_train_model`),
  quindi non serve riaddestrare la CNN a ogni esecuzione.
- `extract_maze` non stampa più decine di `plt.show()` per ogni singola
  cella (nell'originale, per un labirinto 8x8 venivano aperte 64 finestre
  con `plt.imshow`): questo rendeva lo script inutilizzabile in modo
  automatico/non interattivo. Ora il debug visivo è opzionale
  (`debug=True`) e disegna una griglia unica invece di un plot per cella.
- Le soglie di probabilità "magiche" (`prob_S > 0.05`, `prob_T > 0.05`)
  che nell'originale sovrascrivevano la predizione del modello quasi a
  caso sono state rimosse: si usa semplicemente la classe con
  probabilità massima (`argmax`), che è già la predizione della rete.
- Viene validato che la matrice estratta contenga esattamente una 'S' e
  almeno una 'T', con un errore chiaro se non è così (il progetto
  richiede esplicitamente questo vincolo, capitolo 4.1 del pdf).
"""
from __future__ import annotations

import os
from typing import List, Tuple

import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer

CELL_SIZE = 28
WEIGHTS_PATH = "maze_classifier.weights.h5"
CLASSES_PATH = "maze_classifier.classes.npy"


def _lazy_import_tf():
    """Importa tensorflow solo quando serve, per non rendere l'intero
    pacchetto inutilizzabile (es. per chi vuole solo la parte di ricerca)
    se tensorflow non è installato."""
    try:
        import tensorflow as tf  # noqa: F401
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input
        from tensorflow.keras.optimizers import Adam
        from tensorflow.keras.callbacks import EarlyStopping
    except ImportError as exc:
        raise ImportError(
            "tensorflow non è installato. Esegui: pip install tensorflow"
        ) from exc
    return Sequential, Conv2D, MaxPooling2D, Flatten, Dense, Input, Adam, EarlyStopping


def load_training_data(data_path: str):
    """Carica le immagini 28x28 di training, organizzate in
    sottocartelle: una per ciascuna classe ('S', 'T', 'X', '1'..'4')."""
    images, labels = [], []
    classes = sorted(
        d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d))
    )
    if not classes:
        raise ValueError(f"Nessuna sottocartella di classe trovata in '{data_path}'.")

    for label in classes:
        class_path = os.path.join(data_path, label)
        for img_name in os.listdir(class_path):
            img = cv2.imread(os.path.join(class_path, img_name), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (CELL_SIZE, CELL_SIZE)) / 255.0
            images.append(img)
            labels.append(label)

    if not images:
        raise ValueError(f"Nessuna immagine valida trovata in '{data_path}'.")

    label_binarizer = LabelBinarizer()
    label_binarizer.fit(classes)

    images = np.array(images).reshape(-1, CELL_SIZE, CELL_SIZE, 1)
    labels = label_binarizer.transform(labels)
    return images, labels, label_binarizer


def build_model(num_classes: int):
    Sequential, Conv2D, MaxPooling2D, Flatten, Dense, Input, Adam, _ = _lazy_import_tf()
    model = Sequential([
        Input(shape=(CELL_SIZE, CELL_SIZE, 1)),
        Conv2D(32, (3, 3), activation="relu"),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation="relu"),
        Dense(num_classes, activation="softmax"),
    ])
    model.compile(optimizer=Adam(), loss="categorical_crossentropy", metrics=["accuracy"])
    return model


def train_model(data_path: str, epochs: int = 14, batch_size: int = 32):
    """Addestra la CNN sulle immagini di training e salva pesi + classi
    su disco, così le esecuzioni successive possono limitarsi a
    ricaricarli con `load_or_train_model`."""
    _, _, _, _, _, _, _, EarlyStopping = _lazy_import_tf()

    images, labels, label_binarizer = load_training_data(data_path)
    train_images, val_images, train_labels, val_labels = train_test_split(
        images, labels, test_size=0.2, random_state=42
    )

    model = build_model(num_classes=len(label_binarizer.classes_))
    early_stopping = EarlyStopping(monitor="val_loss", patience=5)
    model.fit(
        train_images, train_labels,
        epochs=epochs, batch_size=batch_size,
        validation_data=(val_images, val_labels),
        callbacks=[early_stopping],
    )

    model.save_weights(WEIGHTS_PATH)
    np.save(CLASSES_PATH, label_binarizer.classes_)
    return model, label_binarizer


def load_or_train_model(data_path: str, force_retrain: bool = False, epochs: int = 14):
    """Ricarica il modello già addestrato se presente su disco, altrimenti
    lo addestra da zero (comportamento assente nell'originale, dove ogni
    esecuzione riaddestrava sempre tutto)."""
    if not force_retrain and os.path.exists(WEIGHTS_PATH) and os.path.exists(CLASSES_PATH):
        from sklearn.preprocessing import LabelBinarizer as _LB

        classes = np.load(CLASSES_PATH, allow_pickle=True)
        label_binarizer = _LB()
        label_binarizer.fit(classes)
        model = build_model(num_classes=len(classes))
        model.load_weights(WEIGHTS_PATH)
        return model, label_binarizer

    return train_model(data_path, epochs=epochs)


def extract_maze(image_path: str, model, label_binarizer, debug: bool = False) -> List[List[str]]:
    """Suddivide l'immagine del labirinto in celle 28x28 e classifica
    ciascuna cella con il modello addestrato."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Immagine non trovata o non valida: {image_path}")

    img_height, img_width = img.shape
    if img_height % CELL_SIZE != 0 or img_width % CELL_SIZE != 0:
        raise ValueError(
            f"L'immagine {img_width}x{img_height} non ha dimensioni multiple di "
            f"{CELL_SIZE}px: rigenerala con lo stesso cell_size usato in training."
        )

    num_rows, num_cols = img_height // CELL_SIZE, img_width // CELL_SIZE
    maze_matrix: List[List[str]] = [["" for _ in range(num_cols)] for _ in range(num_rows)]

    for i in range(num_rows):
        for j in range(num_cols):
            y0, x0 = i * CELL_SIZE, j * CELL_SIZE
            cell = img[y0:y0 + CELL_SIZE, x0:x0 + CELL_SIZE]
            cell_input = (cell / 255.0).reshape(1, CELL_SIZE, CELL_SIZE, 1)

            prediction = model.predict(cell_input, verbose=0)
            predicted_label = label_binarizer.classes_[int(np.argmax(prediction, axis=1)[0])]
            maze_matrix[i][j] = str(predicted_label)

    _validate_maze(maze_matrix)

    if debug:
        _draw_debug_grid(img, maze_matrix)

    return maze_matrix


def _validate_maze(maze: List[List[str]]) -> None:
    flat = [cell for row in maze for cell in row]
    n_start = flat.count("S")
    n_treasure = flat.count("T")
    if n_start != 1:
        raise ValueError(
            f"Il labirinto estratto deve contenere esattamente una 'S' (trovate {n_start})."
        )
    if n_treasure < 1:
        raise ValueError("Il labirinto estratto deve contenere almeno un tesoro 'T'.")


def _draw_debug_grid(img: np.ndarray, maze: List[List[str]]) -> None:
    """Disegna in un'unica figura l'esito della classificazione, al posto
    delle decine di finestre separate aperte dall'originale."""
    import matplotlib.pyplot as plt

    rows, cols = len(maze), len(maze[0])
    fig, ax = plt.subplots(figsize=(cols, rows))
    ax.imshow(img, cmap="gray")
    for i in range(rows):
        for j in range(cols):
            ax.text(
                j * CELL_SIZE + CELL_SIZE / 2, i * CELL_SIZE + CELL_SIZE / 2,
                maze[i][j], color="red", ha="center", va="center", fontsize=10, weight="bold",
            )
    ax.set_title("Classificazione celle")
    ax.axis("off")
    plt.show()


## 3. Visualizzazione del percorso trovato

In [ ]:
# -*- coding: utf-8 -*-
"""
Visualizzazione del percorso trovato sull'immagine originale del
labirinto.

Rispetto all'originale, `visualize_solution` ora salva sempre l'immagine
su disco (oltre a poterla mostrare), perché uno script non interattivo
(o eseguito da terminale/CI) non può fare affidamento su `plt.show()`
per produrre un output utilizzabile.
"""
from __future__ import annotations

from typing import List, Tuple

import cv2
import numpy as np

CELL_SIZE = 28
Position = Tuple[int, int]


def visualize_solution(
    image_path: str,
    path: List[Position],
    output_path: str = "maze_solution.png",
    show: bool = False,
    cell_size: int = CELL_SIZE,
) -> str:
    """Disegna il percorso `path` (lista di (riga, colonna)) sull'immagine
    del labirinto in `image_path`, evidenziando le celle attraversate."""
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Immagine non trovata o non valida: {image_path}")

    overlay = img.copy()
    for row, col in path:
        y0, x0 = row * cell_size, col * cell_size
        cv2.rectangle(overlay, (x0, y0), (x0 + cell_size, y0 + cell_size), (0, 0, 255), -1)

    blended = cv2.addWeighted(overlay, 0.45, img, 0.55, 0)
    cv2.imwrite(output_path, blended)

    if show:
        import matplotlib.pyplot as plt

        plt.imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
        plt.title("Labirinto con percorso")
        plt.axis("off")
        plt.show()

    return output_path


## 4. Generatore di labirinti di prova (facoltativo)
Utile per testare la pipeline senza dover disegnare a mano un'immagine. Corregge il bug dell'originale (`generazione_mappa.py`) che non garantiva la presenza di almeno un tesoro.

In [ ]:
# -*- coding: utf-8 -*-
"""
Generatore di immagini di labirinti casuali, utile per testare
`extract_maze` e la pipeline di risoluzione senza dover disegnare a mano
un'immagine.

Bug corretti rispetto all'originale (`generazione_mappa.py`):
- il labirinto generato non conteneva MAI la garanzia di avere almeno un
  tesoro 'T': 'T' era solo una delle 6 possibilità scelte a caso per ogni
  cella di riempimento, quindi capitava spesso di generare labirinti privi
  di tesori, non risolvibili (il progetto richiede sempre almeno un
  tesoro, capitolo 4.1 del pdf).
- l'apertura automatica del file (`os.system("start ...")`) funzionava
  solo su Windows; ora viene rilevata la piattaforma.
"""
from __future__ import annotations

import platform
import random
import subprocess
from typing import List

from PIL import Image, ImageDraw, ImageFont

CELL_SIZE = 28
SYMBOLS = ["1", "2", "3", "4", "X"]


def generate_random_maze(size: int, min_treasures: int = 1) -> List[List[str]]:
    if size < 2:
        raise ValueError("size deve essere almeno 2 (serve spazio per S e almeno un T).")
    max_treasures = max(min_treasures, size - 1)

    maze = [["" for _ in range(size)] for _ in range(size)]
    cells = [(r, c) for r in range(size) for c in range(size)]
    random.shuffle(cells)

    start_cell = cells.pop()
    maze[start_cell[0]][start_cell[1]] = "S"

    n_treasures = random.randint(min_treasures, max_treasures)
    for _ in range(n_treasures):
        r, c = cells.pop()
        maze[r][c] = "T"

    for r, c in cells:
        maze[r][c] = random.choice(SYMBOLS)

    return maze


def generate_maze_image(maze: List[List[str]], cell_size: int = CELL_SIZE, font_size: int = 20) -> Image.Image:
    rows, cols = len(maze), len(maze[0])
    img = Image.new("L", (cols * cell_size, rows * cell_size), color=255)
    draw = ImageDraw.Draw(img)

    for i in range(rows + 1):
        draw.line((0, i * cell_size, cols * cell_size, i * cell_size), fill=0, width=2)
    for j in range(cols + 1):
        draw.line((j * cell_size, 0, j * cell_size, rows * cell_size), fill=0, width=2)

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", font_size)
    except OSError:
        font = ImageFont.load_default()

    for i in range(rows):
        for j in range(cols):
            x, y = j * cell_size + cell_size // 2, i * cell_size + cell_size // 2
            draw.text((x, y), maze[i][j], fill=0, font=font, anchor="mm")

    return img


def save_random_maze_image(size: int, output_path: str = "maze_image.png", open_file: bool = False) -> str:
    maze = generate_random_maze(size)
    img = generate_maze_image(maze)
    img.save(output_path)

    if open_file:
        system = platform.system()
        opener = {"Windows": ["start", output_path], "Darwin": ["open", output_path]}.get(
            system, ["xdg-open", output_path]
        )
        try:
            subprocess.run(opener, shell=(system == "Windows"), check=False)
        except OSError:
            pass

    return output_path


if __name__ == "__main__":
    import sys

    size = int(sys.argv[1]) if len(sys.argv) > 1 else random.randint(4, 8)
    path = save_random_maze_image(size)
    print(f"Labirinto {size}x{size} salvato in: {path}")


## 5. Addestramento del modello
Estrai `maze_data.zip` (da Colab: caricalo quando richiesto) in una cartella `maze_data/` con una sottocartella per classe (`S`, `T`, `X`, `1`, `2`, `3`, `4`), poi esegui l'addestramento. Se `maze_classifier.weights.h5` esiste già, `load_or_train_model` lo ricarica senza riaddestrare.

In [ ]:
if IN_COLAB and not os.path.exists("maze_data"):
    print("Carica maze_data.zip")
    uploaded = files.upload()
    with zipfile.ZipFile("maze_data.zip") as z:
        z.extractall(".")

model, label_binarizer = load_or_train_model("maze_data", epochs=14)
print("Classi:", list(label_binarizer.classes_))

## 6. Estrazione e risoluzione di un labirinto da immagine
Carica un'immagine (dallo stesso formato usato in training: celle 28x28) e risolvi con A* e UCS, con confronto di prestazioni corretto (tempo + numero di stati realmente generati).

In [ ]:
if IN_COLAB:
    print("Carica l'immagine del labirinto da risolvere")
    uploaded = files.upload()
    image_path = list(uploaded.keys())[0]
else:
    image_path = "maze_image.png"  # sostituisci con il tuo file

maze = extract_maze(image_path, model, label_binarizer, debug=True)
print("Matrice del labirinto estratta:")
for row in maze:
    print(row)

In [ ]:
# k=None -> raccogliere tutti i tesori; k=2 -> raccoglierne almeno 2
astar_result, ucs_result = solve_treasure_maze(maze, k=None)

for result in (astar_result, ucs_result):
    print(f"\n--- {result.algorithm.upper()} ---")
    print(f"Tempo di esecuzione: {result.time_seconds:.4f} secondi")
    print(f"Costo del percorso: {result.cost}")
    print(f"Celle esplorate (stati generati): {result.states_generated}")
    print(f"Percorso: {result.path}")

if astar_result.path:
    visualize_solution(image_path, astar_result.path, show=True)